# NLP Assignment 04 — Track 1, Option A
## Person 2: Supervised Fine-Tuning (SFT) with LoRA

**Model:** TinyLlama/TinyLlama_v1.1  
**Dataset:** Medical Q&A (from Person 1 — already preprocessed)  
**Platform:** Google Colab (T4 GPU)  
**Goal:** Run 5 SFT trials, evaluate each on 10 medical test prompts, pick best model for Person 3 (DPO)

---
### Instructions Before Running
1. Go to `Runtime → Change runtime type → T4 GPU`
2. Upload these files from Person 1 to Colab (left panel → upload):
   - `test_prompts.json`
   - `baseline_summary.json`
   - `sft_dataset_preview.csv` (or the HuggingFace dataset name if Person 1 used one)
3. Run all cells top to bottom


## 0. Verify GPU

In [1]:
import torch
assert torch.cuda.is_available(), "❌ GPU not enabled! Go to Runtime → Change runtime type → T4 GPU"
print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
print(f"   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

✅ GPU: Tesla T4
   VRAM: 15.6 GB


## 1. Install Dependencies

In [2]:
!pip install -q transformers==4.40.0 trl==0.8.6 peft==0.10.0 \
    datasets accelerate bitsandbytes sacrebleu bert-score
print("✅ Dependencies installed")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.6/137.6 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.0/9.0 MB 63.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 245.2/245.2 kB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 199.1/199.1 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 24.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 58.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 6.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sentence-transformers 5.5.1 req

## 2. Imports & Global Config

In [3]:
import os, json, time, torch
import pandas as pd
from datasets import Dataset
from transformers import (
    AutoTokenizer, AutoModelForCausalLM,
    BitsAndBytesConfig, TrainingArguments
)
from peft import LoraConfig, TaskType
from trl import SFTTrainer
from sacrebleu.metrics import BLEU
from bert_score import score as bert_score_fn

# ── Config ────────────────────────────────────────────────────
BASE_MODEL  = "TinyLlama/TinyLlama_v1.1"
OUTPUT_DIR  = "/content/sft_outputs"
MAX_SEQ_LEN = 512

SYSTEM_PROMPT = (
    "You are an experienced and knowledgeable medical professional. "
    "Provide clear, factual, and helpful medical information."
)

os.makedirs(OUTPUT_DIR, exist_ok=True)
print("✅ Config ready")

✅ Config ready


## 3. Load Test Prompts & Baseline Scores (from Person 1)
These are fixed — we use the same 10 prompts to compare baseline vs every SFT trial.

In [4]:
with open("test_prompts.json") as f:
    TEST_PROMPTS = json.load(f)

with open("baseline_summary.json") as f:
    BASELINE = json.load(f)

print(f"✅ Loaded {len(TEST_PROMPTS)} test prompts")
print(f"\nBaseline scores to beat:")
print(f"  Corpus BLEU    : {BASELINE['corpus_bleu']:.4f}")
print(f"  BERTScore F1   : {BASELINE['mean_bertscore_f1']:.4f}")

REFERENCES = [p["gold_answer"] for p in TEST_PROMPTS]
QUESTIONS  = [p["question"]    for p in TEST_PROMPTS]

✅ Loaded 10 test prompts

Baseline scores to beat:
  Corpus BLEU    : 1.1166
  BERTScore F1   : 0.7788


## 4. Load & Preprocess SFT Dataset (Person 1's Medical Dataset)

In [5]:
# Load from Person 1's CSV
df = pd.read_csv("sft_dataset_preview.csv")
print(f"Raw dataset: {len(df)} rows")
print(f"Columns: {list(df.columns)}")
df.head(2)

Raw dataset: 10000 rows
Columns: ['text', 'instruction', 'response', 'diff']


,text,instruction,response,diff
0,You are an experienced and knowledgeable medic...,What are the biological mechanisms that differ...,"Tetanus is caused by *Clostridium tetani*, a s...",0
1,You are an experienced and knowledgeable medic...,What are the long-term consequences of delayin...,Delaying treatment for severe tooth decay can ...,0


In [6]:
# Load tokenizer first (needed for chat template formatting)
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
tokenizer.pad_token    = tokenizer.eos_token
tokenizer.padding_side = "right"
print("✅ Tokenizer loaded")

def format_sample(row):
    """
    Format dataset row into TinyLlama chat template.
    Dataset columns: instruction, response (system prompt already in 'text' column)
    """
    messages = [
        {"role": "system",    "content": SYSTEM_PROMPT},
        {"role": "user",      "content": str(row["instruction"])},
        {"role": "assistant", "content": str(row["response"])},
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False)


# Use up to 6000 samples (enough, saves compute)
df_subset = df.dropna(subset=["instruction", "response"]).head(6000).reset_index(drop=True)
df_subset["text"] = df_subset.apply(format_sample, axis=1)

# HuggingFace Dataset + train/val split
hf_dataset = Dataset.from_pandas(df_subset[["text"]])
split = hf_dataset.train_test_split(test_size=0.1, seed=42)
train_data = split["train"]
val_data   = split["test"]

print(f"✅ Train: {len(train_data)} | Val: {len(val_data)}")
print("\nSample formatted text (first 300 chars):")
print(train_data[0]["text"][:300])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/776 [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]


No chat template is defined for this tokenizer - using the default template for the LlamaTokenizerFast class. If the default is not appropriate for your model, please set `tokenizer.chat_template` to an appropriate template. See https://huggingface.co/docs/transformers/main/chat_templating for more information.



✅ Tokenizer loaded
✅ Train: 5400 | Val: 600

Sample formatted text (first 300 chars):
<s>[INST] <<SYS>>
You are an experienced and knowledgeable medical professional. Provide clear, factual, and helpful medical information.
<</SYS>>

What are the common causes of disc bulge at the L4-L5 level and how do these affect nerve function? [/INST] Common causes of disc bulge at the L4-L5 lev


## 5. Helper Functions (Inference + Evaluation)

In [7]:
def generate_response(model, tokenizer, question, max_new_tokens=300):
    """Run inference on a single medical question."""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": question},
    ]
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,           # greedy — same as baseline
            pad_token_id=tokenizer.eos_token_id,
        )
    new_tokens = out[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


def evaluate_on_test_set(model, tokenizer, label=""):
    """Generate responses for all 10 prompts, compute BLEU + BERTScore."""
    print(f"\n  Evaluating [{label}] on 10 test prompts...")
    preds = []
    for p in TEST_PROMPTS:
        resp = generate_response(model, tokenizer, p["question"])
        preds.append(resp)

    # BLEU
    bleu_metric = BLEU(effective_order=True)
    bleu_scores = [bleu_metric.sentence_score(pred, [ref]).score
                   for pred, ref in zip(preds, REFERENCES)]
    avg_bleu = sum(bleu_scores) / len(bleu_scores)

    # BERTScore
    _, _, F1 = bert_score_fn(preds, REFERENCES, lang="en", verbose=False)
    avg_bert = F1.mean().item()

    print(f"  ✅ BLEU: {avg_bleu:.4f} | BERTScore F1: {avg_bert:.4f}")
    return {"bleu": avg_bleu, "bertscore_f1": avg_bert,
            "bleu_per_prompt": bleu_scores,
            "bertscore_per_prompt": F1.tolist(),
            "predictions": preds}


print("✅ Helper functions ready")

✅ Helper functions ready


## 6. Trial Configurations

5 trials varying LoRA rank, target modules, learning rate, batch size, and epochs.

| Trial | Rank | Target Modules | LR | Batch | Epochs |
|-------|------|----------------|----|-------|--------|
| 1 | 8 | q_proj, v_proj | 2e-4 | 4 | 1 |
| 2 | 16 | q_proj, v_proj | 2e-4 | 4 | 2 |
| 3 | 16 | q_proj, v_proj, k_proj | 1e-4 | 8 | 2 |
| 4 | 32 | q_proj, v_proj, k_proj, o_proj | 1e-4 | 4 | 3 |
| 5 | 64 | q_proj, v_proj, k_proj, o_proj | 5e-5 | 8 | 3 |

In [8]:
TRIAL_CONFIGS = [
    {"trial_id": 1, "lora_r": 8,  "lora_alpha": 16,  "lora_dropout": 0.05,
     "target_modules": ["q_proj", "v_proj"],
     "lr": 2e-4, "batch_size": 4, "epochs": 1},

    {"trial_id": 2, "lora_r": 16, "lora_alpha": 32,  "lora_dropout": 0.05,
     "target_modules": ["q_proj", "v_proj"],
     "lr": 2e-4, "batch_size": 4, "epochs": 2},

    {"trial_id": 3, "lora_r": 16, "lora_alpha": 32,  "lora_dropout": 0.1,
     "target_modules": ["q_proj", "v_proj", "k_proj"],
     "lr": 1e-4, "batch_size": 8, "epochs": 2},

    {"trial_id": 4, "lora_r": 32, "lora_alpha": 64,  "lora_dropout": 0.05,
     "target_modules": ["q_proj", "v_proj", "k_proj", "o_proj"],
     "lr": 1e-4, "batch_size": 4, "epochs": 3},

    {"trial_id": 5, "lora_r": 64, "lora_alpha": 128, "lora_dropout": 0.05,
     "target_modules": ["q_proj", "v_proj", "k_proj", "o_proj"],
     "lr": 5e-5, "batch_size": 8, "epochs": 3},
]

print(f"✅ {len(TRIAL_CONFIGS)} trial configs loaded")

✅ 5 trial configs loaded


## 7. SFT Trial Runner

In [9]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
print("✅")

✅


In [10]:
import gc

def run_sft_trial(cfg):
    trial_id  = cfg["trial_id"]
    trial_dir = f"{OUTPUT_DIR}/trial_{trial_id}"
    print(f"\n{'='*65}")
    print(f" TRIAL {trial_id} | rank={cfg['lora_r']} | modules={cfg['target_modules']}")
    print(f"          | lr={cfg['lr']} | batch={cfg['batch_size']} | epochs={cfg['epochs']}")
    print(f"{'='*65}")

    # Use bfloat16 — stable like float32 but half the memory, no fp16 scaler issues
    model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL,
        torch_dtype=torch.bfloat16,
        device_map="auto",
        trust_remote_code=True,
    )
    model.config.use_cache = False

    lora_cfg = LoraConfig(
        r=cfg["lora_r"],
        lora_alpha=cfg["lora_alpha"],
        target_modules=cfg["target_modules"],
        lora_dropout=cfg["lora_dropout"],
        bias="none",
        task_type=TaskType.CAUSAL_LM,
    )

    args = TrainingArguments(
        output_dir=trial_dir,
        num_train_epochs=cfg["epochs"],
        per_device_train_batch_size=cfg["batch_size"],
        per_device_eval_batch_size=2,
        gradient_accumulation_steps=4,      # compensates for smaller batch
        learning_rate=cfg["lr"],
        lr_scheduler_type="cosine",
        warmup_ratio=0.05,
        fp16=False,
        bf16=True,                          # bfloat16 training — stable + memory efficient
        logging_steps=50,
        evaluation_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        report_to="none",
        dataloader_pin_memory=False,
        gradient_checkpointing=True,        # trades compute for memory — key for trials 3-5
    )

    trainer = SFTTrainer(
        model=model,
        args=args,
        train_dataset=train_data,
        eval_dataset=val_data,
        tokenizer=tokenizer,
        peft_config=lora_cfg,
        dataset_text_field="text",
        max_seq_length=MAX_SEQ_LEN,
    )

    trainer.model.print_trainable_parameters()

    t0 = time.time()
    trainer.train()
    elapsed = (time.time() - t0) / 60

    eval_result = trainer.evaluate()
    val_loss = eval_result["eval_loss"]
    print(f"  Val Loss: {val_loss:.4f} | Time: {elapsed:.1f} min")

    scores = evaluate_on_test_set(trainer.model, tokenizer, label=f"Trial {trial_id}")

    trainer.save_model(trial_dir)
    tokenizer.save_pretrained(trial_dir)

    del model, trainer
    torch.cuda.empty_cache()
    gc.collect()

    return {
        "trial_id": trial_id,
        "config": cfg,
        "val_loss": round(val_loss, 4),
        "train_time_min": round(elapsed, 1),
        "bleu": round(scores["bleu"], 4),
        "bertscore_f1": round(scores["bertscore_f1"], 4),
        "bleu_per_prompt": scores["bleu_per_prompt"],
        "bertscore_per_prompt": scores["bertscore_per_prompt"],
        "predictions": scores["predictions"],
        "checkpoint_dir": trial_dir,
    }

print("✅ Ready")

✅ Ready


## 8. Run All 5 Trials

⏱️ **Expected time on Colab T4:** ~15–25 min per trial → ~2 hours total.  
💡 **Tip:** If Colab disconnects, re-run from the last completed trial only — results are saved per trial.

In [ ]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# Load Trials 1 & 2 results — don't redo them
with open("/content/sft_results_so_far.json") as f:
    all_results = json.load(f)
print(f"Loaded {len(all_results)} completed trials")

# Reduce batch size for trials 3-5 to avoid OOM
TRIAL_CONFIGS_REMAINING = [
    {"trial_id": 3, "lora_r": 16, "lora_alpha": 32, "lora_dropout": 0.1,
     "target_modules": ["q_proj", "v_proj", "k_proj"],
     "lr": 1e-4, "batch_size": 2, "epochs": 1},   # 2→1 epoch

    {"trial_id": 4, "lora_r": 32, "lora_alpha": 64, "lora_dropout": 0.05,
     "target_modules": ["q_proj", "v_proj", "k_proj", "o_proj"],
     "lr": 1e-4, "batch_size": 2, "epochs": 1},   # 3→1 epoch

    {"trial_id": 5, "lora_r": 64, "lora_alpha": 128, "lora_dropout": 0.05,
     "target_modules": ["q_proj", "v_proj", "k_proj", "o_proj"],
     "lr": 5e-5, "batch_size": 1, "epochs": 1},   # 3→1 epoch
]

for cfg in TRIAL_CONFIGS_REMAINING:
    result = run_sft_trial(cfg)
    all_results.append(result)
    with open("/content/sft_results_so_far.json", "w") as f:
        json.dump(all_results, f, indent=2)
    print(f"  Progress saved ({len(all_results)}/5 done)\n")

print("\n🎉 All 5 trials complete!")

Loaded 2 completed trials

 TRIAL 3 | rank=16 | modules=['q_proj', 'v_proj', 'k_proj']
          | lr=0.0001 | batch=2 | epochs=1


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Map:   0%|          | 0/5400 [00:00<?, ? examples/s]

Map:   0%|          | 0/600 [00:00<?, ? examples/s]

trainable params: 3,063,808 || all params: 1,103,112,192 || trainable%: 0.2777421936063599


...

Map:   0%|          | 0/5400 [00:00<?, ? examples/s]

Map:   0%|          | 0/600 [00:00<?, ? examples/s]

trainable params: 18,022,400 || all params: 1,118,070,784 || trainable%: 1.6119194113563386


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Epoch,Training Loss,Validation Loss


In [11]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# Load Trials 1-3 — don't redo them
with open("/content/sft_results_so_far.json") as f:
    all_results = json.load(f)
print(f"Loaded {len(all_results)} trials: {[r['trial_id'] for r in all_results]}")

TRIAL_CONFIGS_REMAINING = [
    {"trial_id": 4, "lora_r": 32, "lora_alpha": 64, "lora_dropout": 0.05,
     "target_modules": ["q_proj", "v_proj", "k_proj", "o_proj"],
     "lr": 1e-4, "batch_size": 2, "epochs": 1},

    {"trial_id": 5, "lora_r": 64, "lora_alpha": 128, "lora_dropout": 0.05,
     "target_modules": ["q_proj", "v_proj", "k_proj", "o_proj"],
     "lr": 5e-5, "batch_size": 1, "epochs": 1},
]

for cfg in TRIAL_CONFIGS_REMAINING:
    result = run_sft_trial(cfg)
    all_results.append(result)
    with open("/content/sft_results_so_far.json", "w") as f:
        json.dump(all_results, f, indent=2)
    print(f"  Progress saved ({len(all_results)}/5 done)\n")

print("\n🎉 Trials 4 and 5 complete!")

Loaded 3 trials: [1, 2, 3]

 TRIAL 4 | rank=32 | modules=['q_proj', 'v_proj', 'k_proj', 'o_proj']
          | lr=0.0001 | batch=2 | epochs=1


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


config.json:   0%|          | 0.00/560 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/4.40G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/129 [00:00<?, ?B/s]

...

Epoch,Training Loss,Validation Loss
1,1.261900,1.329780


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


  Val Loss: 1.3298 | Time: 73.9 min

  Evaluating [Trial 5] on 10 test prompts...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


  ✅ BLEU: 3.3385 | BERTScore F1: 0.8292
  Progress saved (5/5 done)


🎉 Trials 4 and 5 complete!


In [ ]:
# Load already completed trials
with open("/content/sft_results_so_far.json") as f:
    all_results = json.load(f)

print(f"Loaded {len(all_results)} completed trials, resuming from Trial 3...")

for cfg in TRIAL_CONFIGS[2:]:   # starts at index 2 = Trial 3
    result = run_sft_trial(cfg)
    all_results.append(result)
    with open("/content/sft_results_so_far.json", "w") as f:
        json.dump(all_results, f, indent=2)
    print(f"  Progress saved ({len(all_results)}/5 done)\n")

print("\n🎉 All 5 trials complete!")

Loaded 2 completed trials, resuming from Trial 3...

 TRIAL 3 | rank=16 | modules=['q_proj', 'v_proj', 'k_proj']
          | lr=0.0001 | batch=8 | epochs=2


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/accelerate/utils/modeling.py:1566: UserWarning: Current model requires 11535744 bytes of buffer for offloaded layers, which seems does not fit any GPU's remaining memory. If you are experiencing a OOM later, please consider using offload_buffers=True.
  warnings.warn(


Map:   0%|          | 0/5400 [00:00<?, ? examples/s]

Map:   0%|          | 0/600 [00:00<?, ? examples/s]

trainable params: 3,063,808 || all params: 1,103,112,192 || trainable%: 0.2777421936063599


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


OutOfMemoryError: CUDA out of memory. Tried to allocate 32.00 MiB. GPU 0 has a total capacity of 14.56 GiB of which 15.81 MiB is free. Including non-PyTorch memory, this process has 14.54 GiB memory in use. Of the allocated memory 14.11 GiB is allocated by PyTorch, and 302.09 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)

## 9. Compare All Trials + Select Best Model

In [12]:
# Build summary table
rows = []
for r in all_results:
    rows.append({
        "Trial"          : r["trial_id"],
        "LoRA Rank"      : r["config"]["lora_r"],
        "Target Modules" : ", ".join(r["config"]["target_modules"]),
        "LR"             : r["config"]["lr"],
        "Batch Size"     : r["config"]["batch_size"],
        "Epochs"         : r["config"]["epochs"],
        "Val Loss"       : r["val_loss"],
        "BLEU"           : r["bleu"],
        "BERTScore F1"   : r["bertscore_f1"],
        "Time (min)"     : r["train_time_min"],
    })

# Add baseline row
baseline_row = {
    "Trial": "Baseline", "LoRA Rank": "-", "Target Modules": "-",
    "LR": "-", "Batch Size": "-", "Epochs": "-", "Val Loss": "-",
    "BLEU": round(BASELINE["mean_sentence_bleu"], 4),
    "BERTScore F1": round(BASELINE["mean_bertscore_f1"], 4),
    "Time (min)": 0,
}

summary_df = pd.DataFrame([baseline_row] + rows)
print(summary_df.to_string(index=False))
summary_df.to_csv("/content/sft_results_summary.csv", index=False)
print("\n✅ Summary saved to sft_results_summary.csv")

   Trial LoRA Rank                 Target Modules       LR Batch Size Epochs Val Loss   BLEU  BERTScore F1  Time (min)
Baseline         -                              -        -          -      -        - 1.1342        0.7788         0.0
       1         8                 q_proj, v_proj   0.0002          4      1   2.7523 1.0196        0.7756        10.4
       2        16                 q_proj, v_proj   0.0002          4      2   2.7514 1.0922        0.7810        20.9
       3        16         q_proj, v_proj, k_proj   0.0001          2      1   1.4113 3.2814        0.8239        75.7
       4        32 q_proj, v_proj, k_proj, o_proj   0.0001          2      1   1.3253 3.2653        0.8268        80.3
       5        64 q_proj, v_proj, k_proj, o_proj  0.00005          1      1   1.3298 3.3385        0.8292        73.9

✅ Summary saved to sft_results_summary.csv


In [13]:
# Selection rule (per assignment): highest BERTScore F1 → then BLEU → then lowest val loss
best = max(all_results, key=lambda r: (r["bertscore_f1"], r["bleu"], -r["val_loss"]))

BEST_SFT_CHECKPOINT = best["checkpoint_dir"]

print("\n" + "="*55)
print(f"  ✅ BEST SFT MODEL: Trial {best['trial_id']}")
print(f"  BERTScore F1 : {best['bertscore_f1']}")
print(f"  BLEU         : {best['bleu']}")
print(f"  Val Loss     : {best['val_loss']}")
print(f"  Config       : rank={best['config']['lora_r']}, "
      f"lr={best['config']['lr']}, "
      f"batch={best['config']['batch_size']}, "
      f"epochs={best['config']['epochs']}")
print(f"  Checkpoint   : {BEST_SFT_CHECKPOINT}")
print("="*55)
print("\n→ Give this checkpoint folder to Person 3 for DPO fine-tuning.")


  ✅ BEST SFT MODEL: Trial 5
  BERTScore F1 : 0.8292
  BLEU         : 3.3385
  Val Loss     : 1.3298
  Config       : rank=64, lr=5e-05, batch=1, epochs=1
  Checkpoint   : /content/sft_outputs/trial_5

→ Give this checkpoint folder to Person 3 for DPO fine-tuning.


## 10. Sample Output Comparison (Baseline vs Best SFT)

In [16]:
# Load baseline responses from Person 1's CSV
baseline_df = pd.read_csv("baseline_results.csv") if os.path.exists("baseline_results.csv") else None

print("SAMPLE OUTPUT COMPARISON")
print("="*70)
for i in range(3):   # show first 3 prompts
    p = TEST_PROMPTS[i]
    sft_pred = best["predictions"][i]

    print(f"\nPrompt {p['id']} [{p['category']}]:")
    print(f"  Q: {p['question']}")
    print(f"\n  [Gold Answer] : {p['gold_answer'][:150]}...")
    if baseline_df is not None:
        base_resp = str(baseline_df.iloc[i].get("response", "N/A"))[:150]
        print(f"  [Baseline]    : {base_resp}...")
    print(f"  [Best SFT]    : {sft_pred[:150]}...")
    print("-"*70)

SAMPLE OUTPUT COMPARISON

Prompt 1 [Chronic Disease]:
  Q: What are the early warning signs of Type 2 diabetes and how is it diagnosed?

  [Gold Answer] : Early warning signs of type 2 diabetes can be subtle and include increased thirst, frequent urination, fatigue, blurred vision, slow-healing cuts/infe...
  [Best SFT]    : Early warning signs of Type 2 diabetes include frequent urination, increased thirst, and weight loss. Diabetes is diagnosed by blood tests and a physi...
----------------------------------------------------------------------

Prompt 2 [Pharmacology]:
  Q: What is the mechanism of action of metformin and why is it the first-line treatment for Type 2 diabetes?

  [Gold Answer] : Metformin mainly reduces hepatic gluconeogenesis and improves peripheral insulin sensitivity, partly through effects on cellular energy metabolism/AMP...
  [Best SFT]    : Metformin is a sulfonylurea-like agent that acts by inhibiting glucose-dependent insulin secretion, thereby reducing gluc

## 11. Save All Results for Report

In [17]:
# Save full results JSON (useful for report tables)
final_output = {
    "baseline": {
        "bleu": BASELINE["mean_sentence_bleu"],
        "bertscore_f1": BASELINE["mean_bertscore_f1"]
    },
    "sft_trials": [
        {
            "trial_id": r["trial_id"],
            "config": r["config"],
            "val_loss": r["val_loss"],
            "bleu": r["bleu"],
            "bertscore_f1": r["bertscore_f1"],
            "train_time_min": r["train_time_min"],
        }
        for r in all_results
    ],
    "best_trial_id": best["trial_id"],
    "best_checkpoint": BEST_SFT_CHECKPOINT,
}

with open("/content/sft_final_results.json", "w") as f:
    json.dump(final_output, f, indent=2)

print("✅ Files ready to download from Colab:")
print("   /content/sft_results_summary.csv  ← for report table")
print("   /content/sft_final_results.json   ← for Person 3 / report")
print(f"   {BEST_SFT_CHECKPOINT}/             ← give to Person 3 for DPO")

✅ Files ready to download from Colab:
   /content/sft_results_summary.csv  ← for report table
   /content/sft_final_results.json   ← for Person 3 / report
   /content/sft_outputs/trial_5/             ← give to Person 3 for DPO


In [18]:
from google.colab import files

# Download results JSON (most important)
files.download("/content/sft_results_so_far.json")

# Download best model checkpoint (run after trials complete)
import shutil
shutil.make_archive("/content/best_checkpoint", "zip", "/content/sft_outputs")
files.download("/content/best_checkpoint.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## ✅ Report Checklist — SFT Section (Your Part)

Copy these results into the report:

- [ ] **Platform:** Google Colab, T4 GPU
- [ ] **Dataset:** Medical Q&A — state source, total rows, subset used (6000), 90/10 split
- [ ] **Table:** 5 trials × (rank, modules, LR, batch, epochs, BLEU, BERTScore F1, val loss, time)
- [ ] **Baseline row** in the same table for comparison
- [ ] **Justification:** why Trial X was selected (highest BERTScore F1, then BLEU, then val loss)
- [ ] **Sample outputs:** 3 prompts showing Baseline vs Best SFT response
- [ ] **Observations:** what improved? what still fails? any unexpected behavior?
- [ ] **LoRA parameter count** (printed by `trainer.model.print_trainable_parameters()`)

---
**Hand off to Person 3:**
- `sft_final_results.json`
- Best checkpoint folder (zipped from `/content/sft_outputs/trial_X/`)
